# 3 – Erweiterter RAG-Graph: Hybrid Search, Reranking, Relevanz-Gate & Query-Reformulierung

- **Hybrid Search** – zwei Retriever (semantisch + lexikalisch) suchen parallel
- **Reranking** – ein Cross-Encoder bewertet die gefundenen Chunks nochmal neu
- **Relevanz-Gate** – eine bedingte Kante entscheidet: *Sind die Chunks gut genug?*
- **Query-Reformulierung** – wenn nicht, wird die Frage umformuliert und erneut gesucht

Der Graph hat jetzt **bedingte Kanten** und einen **Zyklus** – Dinge, die eine lineare Chain nicht kann.

```
retrieve (Hybrid: BM25 + Vektor) → rerank → check_relevance ─(relevant)─→ generate → END
                                                   │
                                             (nicht relevant)
                                                   │
                                                   ▼
                                              reformulate ──→ retrieve  (Zyklus, max. 2×)
```

> **Voraussetzung:** Notebook 1 (Indexing) muss vorher ausgeführt worden sein –
> es erstellt sowohl die Vektordatenbank als auch den BM25-Index.

## Imports

In [ ]:
import os
import base64
import getpass
import pickle
from typing import List, TypedDict

import gradio as gr
from IPython import display

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, END

# Cross-Encoder für Reranking
from sentence_transformers import CrossEncoder

# NEU: BM25-Retriever & Ensemble-Retriever für Hybrid Search
from langchain_classic.retrievers import BM25Retriever
from langchain_classic.retrievers.ensemble import EnsembleRetriever

## Embedding-Modell & Vektordatenbank

In [ ]:
EMBEDDING_MODEL = "intfloat/multilingual-e5-large-instruct"
MODEL_PATH      = "./models"
DB_DIR          = "./chroma_db"
BM25_DIR        = "./bm25_index"


class E5Embeddings(HuggingFaceEmbeddings):
    """Wrapper, der die vom E5-Modell erwarteten Präfixe automatisch setzt."""

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return super().embed_documents(["passage: " + t for t in texts])

    def embed_query(self, text: str) -> list[float]:
        return super().embed_query("query: " + text)


embeddings = E5Embeddings(
    model_name=EMBEDDING_MODEL,
    cache_folder=MODEL_PATH,
)

vectorstore = Chroma(
    persist_directory=DB_DIR,
    embedding_function=embeddings,
)

print(f"✅ Vektordatenbank geladen – {vectorstore._collection.count()} Chunks verfügbar.")

## NEU: BM25-Index laden & Ensemble-Retriever aufbauen

Hier laden wir den in Notebook 1 persistierten BM25-Index und kombinieren ihn
mit dem Vektor-Retriever zu einem **Ensemble-Retriever**.

Der `EnsembleRetriever` von LangChain verwendet intern **Reciprocal Rank Fusion (RRF)**,
um die Ergebnisse beider Retriever zu einem einheitlichen Ranking zu verschmelzen:

$$\text{RRF}(d) = \sum_{r \in R} \frac{1}{k + \text{rank}_r(d)}$$

Dabei ist $k$ ein Glättungsparameter (Standard: 60) und $\text{rank}_r(d)$ die Position
des Dokuments $d$ im Ranking des Retrievers $r$. Das Verfahren ist robust und
funktioniert gut, auch wenn die Scores der einzelnen Retriever nicht direkt vergleichbar sind.

### Gewichtung

Die `weights` steuern, wie stark jeder Retriever in das Endergebnis einfließt:

| Gewichtung | Effekt |
|---|---|
| `[0.5, 0.5]` | Beide gleich gewichtet – guter Startpunkt |
| `[0.4, 0.6]` | Mehr Gewicht auf BM25 – gut bei vielen Fachbegriffen |
| `[0.7, 0.3]` | Mehr Gewicht auf Vektor – gut bei umgangssprachlichen Fragen |

> **Tipp:** Experimentiere mit den Gewichten anhand deiner Query-Sammlung!

In [ ]:
# --- BM25-INDEX LADEN ---

bm25_index_path = os.path.join(BM25_DIR, "chunks.pkl")

with open(bm25_index_path, "rb") as f:
    bm25_chunks = pickle.load(f)

print(f"BM25-Index geladen – {len(bm25_chunks)} Chunks")

In [ ]:
# --- ENSEMBLE-RETRIEVER AUFBAUEN ---

# Wie viele Kandidaten soll jeder einzelne Retriever liefern?
RETRIEVER_K = 15

# Gewichtung: [Vektor, BM25] – Summe sollte 1.0 ergeben
ENSEMBLE_WEIGHTS = [0.5, 0.5]

# 1) Vektor-Retriever (semantische Suche)
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": RETRIEVER_K})

# 2) BM25-Retriever (lexikalische Suche)
bm25_retriever = BM25Retriever.from_documents(bm25_chunks, k=RETRIEVER_K)

# 3) Ensemble: kombiniert beide via Reciprocal Rank Fusion
ensemble_retriever = EnsembleRetriever(
    retrievers=[vector_retriever, bm25_retriever],
    weights=ENSEMBLE_WEIGHTS,
)

print(f"Ensemble-Retriever bereit (Vektor: {ENSEMBLE_WEIGHTS[0]}, BM25: {ENSEMBLE_WEIGHTS[1]})")
print(f"   Jeder Retriever liefert bis zu {RETRIEVER_K} Kandidaten.")

## Reranking-Modell laden

Ein **Bi-Encoder** (unser E5-Modell) ist schnell, weil Query und Dokument getrennt eingebettet werden.  
Ein **Cross-Encoder** ist langsamer, aber genauer: er bewertet Query und Dokument *gemeinsam*.

Strategie: Erst viele Kandidaten mit dem Ensemble-Retriever holen, dann mit dem Cross-Encoder die besten auswählen.

Diese dreistufige Architektur – **breit suchen (Ensemble) → intelligent filtern (Reranker) → antworten (LLM)** –
ist ein bewährtes Muster in modernen RAG-Systemen.

> **Wichtig:** Da unsere Dokumente auf Deutsch sind, brauchen wir einen **multilingualen** Cross-Encoder.  
> Ein rein englischer Reranker (z.B. `ms-marco-MiniLM`) liefert bei deutschen Texten systematisch zu niedrige Scores.

In [ ]:
RERANKER_MODEL = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"

reranker = CrossEncoder(RERANKER_MODEL, max_length=512)

print(f"Reranker geladen: {RERANKER_MODEL}")

## LLM konfigurieren

In [ ]:
DEEPINFRA_API_KEY = getpass.getpass("DeepInfra API-Key eingeben: ")

llm = ChatOpenAI(
    model_name="meta-llama/Llama-3.3-70B-Instruct-Turbo",
    openai_api_key=DEEPINFRA_API_KEY,
    openai_api_base="https://api.deepinfra.com/v1/openai",
    max_tokens=5000,
    temperature=0,
)

## Konfiguration des erweiterten Graphen

In [ ]:
# --- PARAMETER ---

RERANK_TOP_N        = 8       # Die N besten Chunks nach Reranking behalten
RELEVANCE_THRESHOLD = 0.3     # Mindest-Score des besten Chunks (Cross-Encoder)
MAX_RETRIES         = 3       # Maximale Query-Reformulierungen, bevor wir aufgeben

## State – jetzt mit zusätzlichen Feldern

Gegenüber Notebook 2 kommen hinzu:
- `rerank_scores` – die Bewertungen des Cross-Encoders
- `retry_count` – zählt, wie oft die Query reformuliert wurde (Endlosschleifen verhindern!)

In [ ]:
import operator
from typing import List, TypedDict, Annotated

class GraphState(TypedDict):
    question:      str
    context:       List[str]
    metadata:      List[dict]
    rerank_scores: List[float]
    answer:        str
    token_usage:   dict
    retry_count:   int
    query_history: Annotated[List[str], operator.add]   # ← Reducer

## Nodes – die Bausteine des Graphen

Vier Knoten statt zwei:

| Node | Aufgabe |
|------|------|
| `retrieve` | Holt Kandidaten-Chunks via **Hybrid Search** (Vektor + BM25) |
| `rerank` | Bewertet die Kandidaten mit dem Cross-Encoder und behält die besten |
| `generate` | Erzeugt die Antwort mit dem LLM |
| `reformulate` | Formuliert die Frage um, wenn der Kontext nicht relevant genug war |

In [ ]:
# --- NODE: RETRIEVE (HYBRID SEARCH) ---

def retrieve(state: GraphState) -> dict:
    """Holt Kandidaten-Chunks über den Ensemble-Retriever (Vektor + BM25).
    
    Der EnsembleRetriever führt beide Suchen parallel aus und verschmilzt
    die Ergebnisse via Reciprocal Rank Fusion. Duplikate werden automatisch
    entfernt – ein Chunk, der von beiden Retrievern gefunden wird, erhält
    einen höheren RRF-Score.
    """
    print(f"--- RETRIEVE / HYBRID SEARCH (Versuch {state.get('retry_count', 0) + 1}) ---")
    print(f"    Query: {state['question']}")

    docs = ensemble_retriever.invoke(state["question"])

    context  = []
    metadata = []

    for i, doc in enumerate(docs):
        context.append(doc.page_content)
        source_file = os.path.basename(doc.metadata.get("source", "Unbekannt"))
        page_num    = doc.metadata.get("page", 0) + 1
        metadata.append({"id": i + 1, "source": source_file, "page": page_num})

    print(f"    → {len(docs)} Kandidaten nach Fusion (Duplikate entfernt)")
    return {"context": context, "metadata": metadata}

In [ ]:
# --- NODE: RERANK ---

def rerank(state: GraphState) -> dict:
    """Bewertet die Kandidaten mit einem Cross-Encoder und behält die Top-N."""
    print("--- RERANK ---")
    question = state["question"]

    # Cross-Encoder erwartet Paare aus (Query, Passage)
    pairs = [(question, chunk) for chunk in state["context"]]
    scores = reranker.predict(pairs)

    # Nach Score sortieren (absteigend) und Top-N behalten
    scored_items = sorted(
        zip(scores, state["context"], state["metadata"]),
        key=lambda x: x[0],
        reverse=True,
    )

    top_items = scored_items[:RERANK_TOP_N]

    # IDs neu vergeben (1-basiert)
    reranked_context  = []
    reranked_metadata = []
    reranked_scores   = []

    for new_id, (score, text, meta) in enumerate(top_items, start=1):
        reranked_context.append(text)
        reranked_metadata.append({**meta, "id": new_id})
        reranked_scores.append(float(score))

    print(f"    → Top-{RERANK_TOP_N} Scores: {[f'{s:.3f}' for s in reranked_scores]}")

    return {
        "context":       reranked_context,
        "metadata":      reranked_metadata,
        "rerank_scores": reranked_scores,
    }

In [ ]:
# --- NODE: GENERATE ---

ANSWER_PROMPT = ChatPromptTemplate.from_template("""\
Du bist ein präziser Assistent. Beantworte die Frage NUR basierend auf dem KONTEXT.

REGELN:
1. Verweise im Text deiner Antwort auf die Abschnitte, z.B. [1] oder [Quelle: Datei.pdf, S. 5].
2. Wenn die Info nicht im Kontext ist, sag es offen.
3. Erfinde KEINE Fakten.

KONTEXT:
{context}

FRAGE: {question}
""")


def generate(state: GraphState) -> dict:
    """Erzeugt eine Antwort auf Basis der besten Kontext-Chunks."""
    print("--- GENERATE ---")

    formatted_context = ""
    for i, text in enumerate(state["context"]):
        meta  = state["metadata"][i]
        score = state["rerank_scores"][i]
        formatted_context += (
            f"\n--- ABSCHNITT {meta['id']} "
            f"(Quelle: {meta['source']}, Seite {meta['page']}, "
            f"Relevanz: {score:.3f}) ---\n"
            f"{text}\n"
        )

    chain    = ANSWER_PROMPT | llm
    response = chain.invoke({"context": formatted_context, "question": state["question"]})
    usage    = response.response_metadata.get("token_usage", {})

    return {"answer": response.content, "token_usage": usage}

In [ ]:
# --- NODE: REFORMULATE ---

REFORMULATE_PROMPT = ChatPromptTemplate.from_template("""\
Die folgende Suchanfrage hat keine ausreichend relevanten Ergebnisse geliefert.
Formuliere die Anfrage um – nutze Synonyme, andere Formulierungen oder zerlege sie
in einen präziseren Kern. Antworte NUR mit der neuen Suchanfrage, ohne Erklärung.

Ursprüngliche Anfrage: {question}
""")

def reformulate(state: GraphState) -> dict:
    """Formuliert die Frage um, damit die nächste Suche bessere Ergebnisse liefert."""
    print("--- REFORMULATE ---")

    chain        = REFORMULATE_PROMPT | llm
    response     = chain.invoke({"question": state["question"]})
    new_question = response.content.strip()

    retry_count = state.get("retry_count", 0) + 1
    print(f"    → Neue Query: '{new_question}' (Versuch {retry_count})")

    return {
        "question": new_question,
        "retry_count": retry_count,
        "query_history": [new_question],       # ← Reducer hängt an, kein append()
    }

## Bedingte Kante – das Herzstück

Hier passiert das, was eine einfache Chain nicht kann:  
**Der Graph entscheidet zur Laufzeit, welchen Weg er nimmt.**

Die Funktion `check_relevance` prüft den besten Reranking-Score:  
- Über dem Schwellwert → weiter zu `generate`  
- Darunter (und noch Versuche übrig) → zurück zu `reformulate` → `retrieve` (Zyklus!)

In [ ]:
# --- CONDITIONAL EDGE ---

def check_relevance(state: GraphState) -> str:
    """Entscheidet, ob der Kontext relevant genug ist oder reformuliert werden muss."""
    best_score  = max(state["rerank_scores"]) if state["rerank_scores"] else 0.0
    retry_count = state.get("retry_count", 0)

    print(f"--- CHECK RELEVANCE ---")
    print(f"    Bester Score: {best_score:.3f} (Schwelle: {RELEVANCE_THRESHOLD})")
    print(f"    Bisherige Versuche: {retry_count} / {MAX_RETRIES}")

    if best_score >= RELEVANCE_THRESHOLD:
        print("    → RELEVANT – weiter zu Generate")
        return "relevant"
    elif retry_count < MAX_RETRIES:
        print("    → NICHT RELEVANT – Query wird reformuliert")
        return "not_relevant"
    else:
        print("    → NICHT RELEVANT, aber max. Versuche erreicht – Generate mit bestem Ergebnis")
        return "relevant"  # Fallback: trotzdem antworten

## Graph zusammenbauen

Jetzt verbinden wir alles. Beachte den Unterschied zu Notebook 2:  
- `add_conditional_edges` statt `add_edge` nach dem Reranking  
- Ein Zyklus: `reformulate → retrieve → rerank → check_relevance → ...`

In [ ]:
# --- GRAPH ZUSAMMENBAUEN ---

workflow = StateGraph(GraphState)

# Knoten registrieren
workflow.add_node("retrieve_node",    retrieve)
workflow.add_node("rerank_node",      rerank)
workflow.add_node("generate_node",    generate)
workflow.add_node("reformulate_node", reformulate)

# Kanten definieren
workflow.set_entry_point("retrieve_node")
workflow.add_edge("retrieve_node", "rerank_node")

# Die bedingte Kante: check_relevance entscheidet den Weg
workflow.add_conditional_edges(
    "rerank_node",
    check_relevance,
    {
        "relevant":     "generate_node",
        "not_relevant": "reformulate_node",
    },
)

# Der Zyklus: reformulate → retrieve (und von dort wieder rerank → check)
workflow.add_edge("reformulate_node", "retrieve_node")
workflow.add_edge("generate_node", END)

# Graph kompilieren
app = workflow.compile()

## Graph visualisieren

Vergleiche dieses Diagramm mit dem aus Notebook 2.  
Hier sieht man den Unterschied: **bedingte Verzweigung und ein Zyklus**.

In [ ]:
def display_graph(graph_app):
    """Zeigt den LangGraph als Mermaid-Diagramm an und speichert die Syntax."""
    mermaid_code = graph_app.get_graph().draw_mermaid()

    encoded = base64.b64encode(mermaid_code.encode()).decode()
    display.display(display.Image(url=f"https://mermaid.ink/img/{encoded}"))

    with open("rag_graph_erweitert.mmd", "w", encoding="utf-8") as f:
        f.write(mermaid_code)


display_graph(app)

## Gradio-Interface

In [ ]:
import os
from urllib.parse import quote

# Absoluter Pfad zum Dokumenten-Ordner
DOCUMENTS_ABS = os.path.abspath("./documents")

def chat_interface(question: str) -> str:
    """Verarbeitet eine Frage über den erweiterten RAG-Graphen mit Hybrid Search."""
    
    yield "⏳ Hybrid Search, Reranking & Antwortgenerierung laufen..."
    
    result = app.invoke({
        "question": question, 
        "retry_count": 0,
        "query_history": [question]
    })

    answer = result["answer"]

    # Reranking-Info MIT klickbaren Dokument-Links
    scores = result.get("rerank_scores", [])
    rerank_info = "\n\n---\n🎯 **Reranking-Scores (Top-Chunks):**\n"
    for i, (meta, score) in enumerate(zip(result["metadata"], scores)):
        source = meta['source']
        page   = meta['page']
        file_path = os.path.join(DOCUMENTS_ABS, source)
        link = f"/gradio_api/file={quote(file_path)}#page={page}"
        rerank_info += f"- [{meta['id']}] [{source} (S. {page})]({link}): {score:.3f}\n"

    # Hybrid-Search-Info
    hybrid_info = (
        f"\n🔀 **Hybrid Search:**\n"
        f"- Retriever: Vektor ({ENSEMBLE_WEIGHTS[0]}) + BM25 ({ENSEMBLE_WEIGHTS[1]})\n"
        f"- Kandidaten pro Retriever: {RETRIEVER_K}\n"
        f"- Fusion: Reciprocal Rank Fusion\n"
    )

    # Token-Statistik
    usage = result.get("token_usage", {})
    token_info = (
        f"\n📊 **Token-Statistik:**\n"
        f"- Input: {usage.get('prompt_tokens', 'N/A')}\n"
        f"- Output: {usage.get('completion_tokens', 'N/A')}\n"
        f"- Gesamt: {usage.get('total_tokens', 'N/A')}"
    )

    # Retry-Info
    retries = result.get("retry_count", 0)
    retry_info = ""
    if retries > 0:
        retry_info = f"\n\n🔄 Query wurde {retries}× reformuliert:\n"
        history = result.get("query_history", [])
        
        for i, q in enumerate(history):
            if i == 0:
                retry_info += f"- **Original:** {q}\n"
            else:
                retry_info += f"- **Versuch {i}:** {q}\n"

    yield answer + rerank_info + hybrid_info + token_info + retry_info


demo = gr.Interface(
    fn=chat_interface,
    inputs="text",
    outputs=gr.Markdown(),
    title="RAG mit Hybrid Search, Reranking & Relevanz-Gate",
    description="Fragen an deine Dokumente – mit Hybrid Search (Vektor + BM25), Cross-Encoder-Reranking und automatischer Query-Reformulierung.",
    flagging_mode="never",
)

demo.launch(allowed_paths=[DOCUMENTS_ABS])